# Population gradient flow from the equivariant init

The 60-unit equivariant init (runs 37-40) is fixed by the tree's symmetry group up to a rotation of the off-span complement, so under
the population gradient its trajectory stays on the symmetric subspace and is described by a few coordinates per class of rows and
units. This notebook characterises that flow with full-batch gradient descent on the 1M-point sample (run 40, one step per
"epoch", constant lr 0.01, momentum 0.9) as the proxy, and uses the SGD runs 37 / 38 from the same init as the finite-batch contrast.

Sections:
1. The symmetric subspace: which first-layer coordinates are free (`symmetric_coordinates`), the residual (what must vanish) along
   run 40 and along the SGD runs.
2. The trajectory in class coordinates: own, ancestor, level-4 coordinates of the probe and cue classes against steps; loss.
3. The population gradient at checkpoints: how symmetric it is on the training sample vs the test sample (sampling noise of the
   gradient itself).
4. The solution: node coverage, unit anatomy and path signs at the end of run 40, vs run 38.

**Findings (2026-10-12, runs 37 / 38 / 40).**
- Full-batch GD (run 40) and SGD with batch 10k (run 38) from the same init follow the same trajectory: the weights agree to 0.1% at
  step 1000 and 0.5% at 10 000, then separate slowly (10-19% at 150k, cosines > 0.98) while residuals, spreads, in-span norms and
  fates stay the same to two or three digits. The sample, not the batch noise, is what breaks the symmetry.
- The sample gradient at the equivariant init deviates from symmetry by about 10% of its size on 1M points (5e-5 vs 4e-4) and twice
  that on 200k; the symmetric part of the dynamics outgrows it, and by step 100k the training-sample gradient is symmetric to 2e-4
  while the test-sample gradient deviates by 1e-2: the solution has fitted the training sample's asymmetry.
- Levels 0-2 follow the symmetric flow to the end (own-coordinate spreads 0.05 / 0.26 / 0.82 on in-span norms 8.6 / 6.2 / 4.8 at
  step 150k); level 3 breaks from step ~10k on (spread 3.7 on norm 2.8 at 150k) and the cues split between their own normal and an
  ancestor. The population-flow solution is therefore characterised by one representative per level for levels 0-2 (own coordinate
  growing, a small positive coefficient on the parent, zero elsewhere) and is not determined at level 3, where the sample decides.


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2
from layer_lenses.designed_init import (symmetric_coordinates, symmetric_class_summary, flip_equivariance_report, cue_row_identity, node_ancestors, NODE_LEVEL, HIERARCHY_TARGETS,
                                        probe_deviation_lineage, unit_leaf_anatomy, tree_flip_matrix)
from layer_lenses.init_sweep import load_tree, SweepRunSpec, build_data, build_data_chunked
from layer_lenses.relu_analysis import load_designed_run, population_gradient_at_snapshot
RESULTS = Path("../results/designed_init")
tree = load_tree(RESULTS / "hierarchy_equivariant_60_nospare" / "tree.npz"); U = tree.w_list
FOLDERS = {37: "hierarchy_equivariant_60", 38: "hierarchy_equivariant_60_nospare", 39: "hierarchy_equivariant_60_nospare_cue01", 40: "hierarchy_equivariant_60_nospare_pgf"}
def load_stacks(n):
    f = RESULTS / FOLDERS[n] / "replay"; z = np.load(f / "weights.npz"); import pickle; meta = pickle.load(open(f / "meta.pkl", "rb"))
    return {"epochs": z["epochs"], "W": {k[3:]: z[k] for k in z.files if k.startswith("w__")}, "losses": np.array(meta["out"]["epoch_losses"]), "design": meta["design"], "steps_per_epoch": 1 if meta["design"].get("full_batch_steps") else 100}
RUNS = {n: load_stacks(n) for n in FOLDERS if (RESULTS / FOLDERS[n] / "replay" / "weights.npz").exists()}
print("loaded:", {n: (r["W"]["hidden_layers.0.weight"].shape, len(r["epochs"])) for n, r in RUNS.items()})

## 1. The symmetric subspace and its residual

`symmetric_coordinates` writes every first-layer row in the coordinates the group leaves free: a probe of level l has `own`, `anc_1..anc_l`
(side-signed ancestor coefficients), `desc4_same` / `desc4_other` (one common coefficient on the level-4 normals under it, on the pole's
side / the other side) and `unrel4_b0..b{l-1}` (one common coefficient on the level-4 normals by the level of the deepest common ancestor);
a cue has `own`, `anc_1..anc_4`, `unrel4_b0..b3`; a spare row only `lvl4`. Everything else in span (`residual`) and every spread inside a
class must vanish on the subspace. The cells below track the residual and the spreads along run 40 (full batch: only the fixed sample breaks
the symmetry) and along runs 37 / 38 (SGD: batch noise too).

In [ ]:
def residual_series(n, steps=(0, 100, 300, 1000, 3000, 10000, 30000, 100000, 150000, 1000000)):
    r = RUNS[n]; cr = r["design"]["cue_rows"]; W1 = r["W"]["hidden_layers.0.weight"]; cid = cue_row_identity(W1[0].astype(np.float64), tree, cr); rows = []
    for st in steps:
        ep = st // r["steps_per_epoch"]
        if ep > r["epochs"][-1]: continue
        i = int(np.searchsorted(r["epochs"], ep)); co = symmetric_coordinates(W1[i].astype(np.float64), tree, cue_rows=cr, cue_identity=cid); summ = symmetric_class_summary(co)
        spreads = summ.filter(like="_spread").abs().max(axis=1)
        rows.append({"run": n, "step": st, "residual_max": co.residual.max(), "residual_probe_mean": co[co.kind == "probe"].residual.mean(), "residual_cue_mean": co[co.kind == "cue"].residual.mean(),
                     "spread_max": spreads.max(), "in_span_probe_L0": summ.set_index("cls").loc["probe_L0", "in_span_norm"], "in_span_cue": summ.set_index("cls").loc["cue_+", "in_span_norm"], "loss": r["losses"][max(ep - 1, 0)]})
    return pd.DataFrame(rows)
res = pd.concat([residual_series(n) for n in RUNS]); print(res.round(4).to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for n, g in res.groupby("run"):
    ax[0].plot(g.step.clip(lower=1), g.residual_max, marker="o", label=f"run {n}"); ax[1].plot(g.step.clip(lower=1), g.spread_max, marker="o", label=f"run {n}")
for a, t in zip(ax, ["largest residual (in-span part that must vanish)", "largest within-class spread of a free coordinate"]): a.set_xscale("log"); a.set_yscale("log"); a.set_xlabel("step"); a.set_title(t); a.legend()
plt.tight_layout(); plt.show()

## 2. The trajectory in class coordinates

In [ ]:
def class_series(n, cls_list=("probe_L0", "probe_L1", "probe_L2", "probe_L3", "cue_+", "cue_-"), every=1):
    r = RUNS[n]; cr = r["design"]["cue_rows"]; W1 = r["W"]["hidden_layers.0.weight"]; cid = cue_row_identity(W1[0].astype(np.float64), tree, cr); rows = []
    for i in range(0, len(r["epochs"]), every):
        summ = symmetric_class_summary(symmetric_coordinates(W1[i].astype(np.float64), tree, cue_rows=cr, cue_identity=cid)).set_index("cls")
        for cls in cls_list:
            rows.append({"step": int(r["epochs"][i]) * r["steps_per_epoch"], "cls": cls, **{k: summ.loc[cls, k] for k in summ.columns if not k.endswith("_spread") and k not in ("n",) and pd.notna(summ.loc[cls, k])}})
    return pd.DataFrame(rows)
N_PGF = max(RUNS)                        # the full-batch run when it exists, else the latest SGD run
cs = class_series(N_PGF, every=5)
fig, ax = plt.subplots(2, 3, figsize=(15, 7)); ax = ax.ravel()
for k, cls in enumerate(["probe_L0", "probe_L1", "probe_L2", "probe_L3", "cue_+", "cue_-"]):
    g = cs[cs.cls == cls].set_index("step")
    for col in [c for c in ["own", "anc_1", "anc_2", "anc_3", "anc_4", "desc4_same", "desc4_other", "unrel4_b0", "unrel4_b1", "unrel4_b2", "unrel4_b3", "off_span_norm"] if c in g.columns]:
        ax[k].plot(np.clip(g.index.values, 1, None), g[col], label=col, lw=1)
    ax[k].set_xscale("log"); ax[k].set_title(f"run {N_PGF}: {cls}"); ax[k].legend(fontsize=7, ncol=2); ax[k].set_xlabel("step")
plt.tight_layout(); plt.show()

In [ ]:
# loss against steps: full batch vs SGD from the same init
plt.figure(figsize=(8, 3))
for n, r in RUNS.items():
    plt.plot(np.arange(1, len(r["losses"]) + 1) * r["steps_per_epoch"], r["losses"], lw=0.7, label=f"run {n} ({'full batch' if r['steps_per_epoch'] == 1 else 'SGD 10k'})")
plt.xscale("log"); plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()

## 3. How symmetric is the population gradient itself?

The sample gradient at a symmetric point is the population gradient (exactly symmetric) plus sampling noise of order 1/sqrt(N). The cell
applies the 15 flips to the gradient at the init and at later checkpoints of run 40, on the 1M training sample and on the 200k test sample.

In [ ]:
x, y = build_data_chunked(SweepRunSpec(tree_seed=5178, data_seed=5178, init_seed=0, batch_size=1, n_train=1_000_000), tree)
xe, ye = build_data(SweepRunSpec(tree_seed=5178, data_seed=999_777, init_seed=0, batch_size=1, n_train=200_000), tree)
r = RUNS[N_PGF]; cr = r["design"]["cue_rows"]; cid0 = cue_row_identity(r["W"]["hidden_layers.0.weight"][0].astype(np.float64), tree, cr)
run_w = {"epochs": r["epochs"], **{"w__" + k: v for k, v in r["W"].items()}}
rows = []
for st in [0, 1000, 10000, 100000]:
    ep = st // r["steps_per_epoch"]
    if ep > r["epochs"][-1]: continue
    for name, (xx, yy) in {"train 1M": (x, y), "test 200k": (xe, ye)}.items():
        g = population_gradient_at_snapshot(run_w, ep, xx, yy, device="cuda")
        G1, G2, G3, Go = (g[k] for k in ["hidden_layers.0.weight", "hidden_layers.1.weight", "hidden_layers.2.weight", "output_layer.weight"])
        reps = {j: flip_equivariance_report(G1, G2, G3, Go, tree, j, cue_rows=cr, units_per_node=4, cue_identity=cid0) for j in range(15)}
        rows.append({"step": st, "sample": name, **{f"max_{k}": max(rp[k] for rp in reps.values()) for k in ["in_span", "off_span_gram", "w2", "w3"]}, "|grad W1|": np.abs(G1).max(), "|grad W2|": np.abs(G2).max(), "|grad W3|": np.abs(G3).max()})
print(pd.DataFrame(rows).to_string(index=False))

## 4. The solution at the end of run 40

In [ ]:
r = RUNS[N_PGF]; W1, W2, W3, Wo = (r["W"][k][-1].astype(np.float64) for k in ["hidden_layers.0.weight", "hidden_layers.1.weight", "hidden_layers.2.weight", "output_layer.weight"])
lin = probe_deviation_lineage(W1, tree); print("probes on own node:", int((~lin.deviated & (lin.end_cos.abs() >= 0.95)).sum()), "| deviations:", lin[lin.deviated][["probe", "target", "end_node", "relation"]].values.tolist(), "| uncaptured:", lin[(lin.end_cos.abs() < 0.95) & ~lin.deviated].probe.tolist())
C = W1 / np.linalg.norm(W1, axis=1, keepdims=True) @ U.T; cr = r["design"]["cue_rows"]; inv = {v: k for k, v in cue_row_identity(r["W"]["hidden_layers.0.weight"][0].astype(np.float64), tree, cr).items()}
print("cues: on own normal", sum(1 for rr in cr if abs(C[rr]).argmax() == inv[rr][0] and abs(C[rr]).max() > 0.95), "| on an ancestor", sum(1 for rr in cr if abs(C[rr]).max() > 0.95 and abs(C[rr]).argmax() in node_ancestors(inv[rr][0])), "| diffuse", sum(1 for rr in cr if abs(C[rr]).max() <= 0.95))
print("nodes covered:", len(set(np.abs(C).argmax(1)[np.abs(C).max(1) > 0.95])), "/ 31")
anat = unit_leaf_anatomy(W1, W2, W3, Wo, tree, xe[:50000]); print(anat.groupby(anat.unit // 4).agg(active=("active", "mean"), n_veto=("n_veto", "mean"), top_share=("share", "mean")).round(2).T)
summ_end = symmetric_class_summary(symmetric_coordinates(W1, tree, cue_rows=cr, cue_identity=cue_row_identity(r["W"]["hidden_layers.0.weight"][0].astype(np.float64), tree, cr))); print(summ_end.round(3).to_string(index=False))